In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, f1_score, top_k_accuracy_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

# 设置随机种子确保可复现
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==================== 1. 主干网络（输出全局+中间层局部特征）====================
class ResNetBackbone(nn.Module):
    def __init__(self, base_encoder):
        super().__init__()
        try:
            self.backbone = base_encoder(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        except AttributeError:
            self.backbone = base_encoder(pretrained=True)

        # CIFAR 小图适配
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()

        self.feature_dim = 2048    # 全局特征维度
        self.local_dim = 1024      # 局部特征维度（layer3 输出）

    def forward(self, x):
        # 前向传播，提取多层特征
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x1 = self.backbone.layer1(x)
        x2 = self.backbone.layer2(x1)
        x3 = self.backbone.layer3(x2)  # 局部特征层
        x4 = self.backbone.layer4(x3)  # 高层特征

        # 全局特征：最终层平均池化
        global_feat = self.backbone.avgpool(x4)
        global_feat = torch.flatten(global_feat, 1)

        # 局部特征：中间层平均池化（强纹理/结构信息）
        local_feat = self.backbone.avgpool(x3)
        local_feat = torch.flatten(local_feat, 1)

        return global_feat, local_feat

# ==================== 2. 投影头（全局/局部各用一个）====================
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=2048, output_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

# ==================== 3. 全局+局部模型 ====================
class SimCLRModel(nn.Module):
    def __init__(self, backbone, proj_global, proj_local):
        super().__init__()
        self.encoder = backbone
        self.proj_global = proj_global
        self.proj_local = proj_local

    def forward(self, x):
        # 同时提取全局 + 局部特征
        global_feat, local_feat = self.encoder(x)

        # 投影 + 归一化
        z_global = self.proj_global(global_feat)
        z_global = nn.functional.normalize(z_global, dim=1)

        z_local = self.proj_local(local_feat)
        z_local = nn.functional.normalize(z_local, dim=1)

        return z_global, z_local, global_feat

# ==================== 4. 数据增强 ====================
class SimCLRTransform:
    def __init__(self, image_size):
        color_jitter = transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
        self.pil_train_transform = transforms.Compose([
            transforms.RandomResizedCrop(size=image_size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
        ])
        self.to_tensor_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.507, 0.487, 0.441],
                                std=[0.267, 0.256, 0.276]),
        ])

    def __call__(self, x):
        from torchvision.transforms.functional import to_pil_image
        if isinstance(x, torch.Tensor):
            x = to_pil_image(x)
        v1 = self.pil_train_transform(x)
        v2 = self.pil_train_transform(x)
        return self.to_tensor_normalize(v1), self.to_tensor_normalize(v2)

# 读取数据集
class CIFAR100WithClassAsCluster(torch.utils.data.Dataset):
    def __init__(self, root, train=True, transform=None, download=True):
        self.base_dataset = datasets.CIFAR100(root=root, train=train, download=download)
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        if self.transform is not None:
            v1, v2 = self.transform(img)
        else:
            v1 = v2 = img
        return v1, v2, label, idx

# ==================== 5. 损失====================
def hard_negative_clustered_simclr_loss(z1, z2, cluster_labels_batch, temperature=0.2, hard_neg_weight=1.5):
    device = z1.device
    batch_size = z1.shape[0]
    z_full = torch.cat((z1, z2), dim=0)
    sim_matrix_full = torch.mm(z_full, z_full.T) / temperature

    pos_mask = torch.zeros((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    pos_mask[torch.arange(batch_size), torch.arange(batch_size) + batch_size] = True
    pos_mask[torch.arange(batch_size) + batch_size, torch.arange(batch_size)] = True

    mask_neg = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    mask_neg.fill_diagonal_(False)
    mask_neg[pos_mask] = False

    cl_labels_expanded = cluster_labels_batch.repeat(2)
    same_cluster_mask_full = (cl_labels_expanded.unsqueeze(0) == cl_labels_expanded.unsqueeze(1))
    cluster_neg_mask = ~same_cluster_mask_full
    cluster_neg_mask[pos_mask] = True

    hard_neg_threshold = 0.5
    hard_neg_mask = (sim_matrix_full > hard_neg_threshold) & mask_neg & cluster_neg_mask

    weight_matrix = torch.ones_like(sim_matrix_full, device=device)
    weight_matrix[hard_neg_mask] = hard_neg_weight

    sim_matrix_weighted = sim_matrix_full * weight_matrix
    final_neg_mask = mask_neg & cluster_neg_mask
    sim_matrix_final = sim_matrix_weighted.masked_fill(~final_neg_mask, float('-inf'))

    pos_sim = torch.sum(z1 * z2, dim=1) / temperature
    pos_sim_full = torch.cat([pos_sim, pos_sim], dim=0)

    exp_sim_neg = torch.exp(sim_matrix_final)
    sum_exp_neg = exp_sim_neg.sum(dim=1)
    exp_pos_sim = torch.exp(pos_sim_full)

    log_prob_pos = pos_sim_full - torch.log(exp_pos_sim + sum_exp_neg)
    loss = -log_prob_pos.mean()

    return loss

# ==================== 6. 全局 + 局部联合损失 ====================
def combined_clustered_loss(z1_g, z2_g, z1_l, z2_l, cluster_labels, temp=0.2, alpha=0.5):
    loss_global = hard_negative_clustered_simclr_loss(z1_g, z2_g, cluster_labels, temp)
    loss_local  = hard_negative_clustered_simclr_loss(z1_l, z2_l, cluster_labels, temp)
    total = alpha * loss_global + (1 - alpha) * loss_local
    return total, loss_global, loss_local

def get_cosine_scheduler(optimizer, epochs, warmup_epochs=10):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def plot_loss_curves(all_losses, save_path='loss_curves.png'):
    plt.figure(figsize=(10,4))
    plt.plot(all_losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

# ==================== 7. 线性评估 ====================
def evaluate_linear(model, train_loader, test_loader, num_classes=100, epochs=50, device='cuda'):
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = nn.Sequential(
        nn.Linear(model.encoder.feature_dim, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(2048, num_classes)
    ).to(device)

    opt = optim.SGD(head.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best = 0.0
    for ep in range(epochs):
        head.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad():
                _, _, f = model(x)
            loss = criterion(head(f), y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        head.eval()
        correct, total = 0,0
        with torch.no_grad():
            for x,y in test_loader:
                x,y = x.to(device), y.to(device)
                _, _, f = model(x)
                pred = head(f).argmax(1)
                correct += (pred==y).sum().item()
                total += y.size(0)
        acc = 100.*correct/total
        if acc>best: best=acc
        print(f"Linear Epoch {ep+1:2d} | Acc={acc:.2f}% | Best={best:.2f}%")
        scheduler.step()
    return best

# ==================== 8. 主函数 ====================
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    batch_size = 512
    lr = 1e-3
    temperature = 0.1
    epochs = 100
    image_size = 32

    train_dataset = CIFAR100WithClassAsCluster(
        root='./data', train=True, transform=SimCLRTransform(image_size), download=True
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)

    # ==================== 模型 ====================
    backbone = ResNetBackbone(torchvision.models.resnet50)
    proj_global = ProjectionHead(input_dim=2048)
    proj_local  = ProjectionHead(input_dim=1024)
    model = SimCLRModel(backbone, proj_global, proj_local).to(device)

    # 优化
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = get_cosine_scheduler(optimizer, epochs)
    scaler = torch.cuda.amp.GradScaler()

    # 训练
    model.train()
    all_losses = []
    for epoch in range(epochs):
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for v1, v2, cls_label, _ in pbar:
            v1, v2, cls_label = v1.to(device), v2.to(device), cls_label.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                z1_g, z1_l, _ = model(v1)
                z2_g, z2_l, _ = model(v2)
                # 联合损失
                loss, loss_g, loss_l = combined_clustered_loss(
                    z1_g, z2_g, z1_l, z2_l, cls_label, temperature
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item(), g=loss_g.item(), l=loss_l.item())

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        all_losses.append(avg_loss)
        print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.3f}")

    plot_loss_curves(all_losses)

    # 测试
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.507,0.487,0.441],[0.267,0.256,0.276])
    ])
    lt = DataLoader(datasets.CIFAR100('./data',train=True,transform=eval_transform), batch_size=512, shuffle=True, num_workers=2)
    lv = DataLoader(datasets.CIFAR100('./data',train=False,transform=eval_transform), batch_size=512, shuffle=False, num_workers=2)

    best_acc = evaluate_linear(model, lt, lv, epochs=100, device=device)
    print(f"\nFINAL BEST ACCURACY: {best_acc:.2f}%")

if __name__ == "__main__":
    main()

Using device: cuda
Files already downloaded and verified


Epoch 1/100: 100%|██████████| 97/97 [00:42<00:00,  2.26it/s, g=7.25, l=6.67, loss=6.96]


Epoch 1 Avg Loss: 7.003


Epoch 2/100: 100%|██████████| 97/97 [00:42<00:00,  2.28it/s, g=3.99, l=3.58, loss=3.78]


Epoch 2 Avg Loss: 4.734


...

Epoch 98 Avg Loss: 0.600


Epoch 99/100: 100%|██████████| 97/97 [00:42<00:00,  2.29it/s, g=0.566, l=0.636, loss=0.601]


Epoch 99 Avg Loss: 0.606


Epoch 100/100: 100%|██████████| 97/97 [00:42<00:00,  2.29it/s, g=0.612, l=0.69, loss=0.651] 


Epoch 100 Avg Loss: 0.609
Linear Epoch  1 | Acc=68.50% | Best=68.50%
Linear Epoch  2 | Acc=70.03% | Best=70.03%
Linear Epoch  3 | Acc=70.83% | Best=70.83%
Linear Epoch  4 | Acc=71.34% | Best=71.34%
Linear Epoch  5 | Acc=71.90% | Best=71.90%
Linear Epoch  6 | Acc=71.27% | Best=71.90%
Linear Epoch  7 | Acc=71.95% | Best=71.95%
Linear Epoch  8 | Acc=72.56% | Best=72.56%
Linear Epoch  9 | Acc=72.46% | Best=72.56%
Linear Epoch 10 | Acc=72.38% | Best=72.56%
Linear Epoch 11 | Acc=73.14% | Best=73.14%
Linear Epoch 12 | Acc=72.75% | Best=73.14%
Linear Epoch 13 | Acc=72.94% | Best=73.14%
Linear Epoch 14 | Acc=73.07% | Best=73.14%
Linear Epoch 15 | Acc=72.90% | Best=73.14%
Linear Epoch 16 | Acc=72.98% | Best=73.14%
Linear Epoch 17 | Acc=73.46% | Best=73.46%
Linear Epoch 18 | Acc=73.33% | Best=73.46%
Linear Epoch 19 | Acc=73.76% | Best=73.76%
Linear Epoch 20 | Acc=73.40% | Best=73.76%
Linear Epoch 21 | Acc=73.68% | Best=73.76%
Linear Epoch 22 | Acc=73.60% | Best=73.76%
Linear Epoch 23 | Acc=73.98%